# Travail de groupe ETL

On commence par importer les librairies nécessaires et charger les fichiers CSV dans un dictionnaire de DataFrames. Ensuite, nous créons une copie du dictionnaire original pour conserver les données brutes.

In [1]:
# Imports des bibliothèques

import glob

import pandas as pd

# afficher toutes les colonnes
pd.set_option("display.max_columns", 50)

all_files = sorted(glob.glob("*.csv"))

df_dic = {}
df_origin = {}
for file in all_files:
    nom = file.replace("olist_", "").replace("_dataset", "").replace(".csv", "")
    df_dic[nom] = pd.read_csv(file)
    df_origin[nom] = pd.read_csv(file)


def resume(df):
    """Renvoie un tableau de synthèse: une ligne par colonne  de df."""
    return pd.DataFrame(
        {
            "type": df.dtypes,
            "nb_uniques": df.nunique(),
            "maximum": df.max(),
            "minimum": df.min(),
            "nb manquants": df.isnull().sum(),
            "pct_manquants": (df.isnull().mean() * 100).round(2),
        }
    )

## Aperçu des tables

In [2]:
for nom, df in df_dic.items():
    print("=====", nom, "=====")
    display(resume(df))

===== customers =====


,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
customer_id,str,99441,ffffe8b65bbe3087b653a978c870db99,00012a2ce6f8dcda20d059ce98491703,0,0.0
customer_unique_id,str,96096,ffffd2657e2aad2907e67c3e9daecbeb,0000366f3b9a7992bf8c76cfdf3221e2,0,0.0
customer_zip_code_prefix,int64,14994,99990,1003,0,0.0
customer_city,str,4119,zortea,abadia dos dourados,0,0.0
customer_state,str,27,TO,AC,0,0.0


===== geolocation =====


,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
geolocation_zip_code_prefix,int64,19015,99990,1001,0,0.0
geolocation_lat,float64,717360,45.065933,-36.605374,0,0.0
geolocation_lng,float64,717613,121.105394,-101.466766,0,0.0
geolocation_city,str,8011,óleo,* cidade,0,0.0
geolocation_state,str,27,TO,AC,0,0.0


===== order_items =====


,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
order_id,str,98666,fffe41c64501cc87c801fd61db3f6244,00010242fe8c5a6d1ba2dd792cb16214,0,0.0
order_item_id,int64,21,21,1,0,0.0
product_id,str,32951,fffe9eeff12fcbd74a2f2b007dde0c58,00066f42aeeb9f3007548bb9d3f33c38,0,0.0
seller_id,str,3095,ffff564a4f9085cd26170f4732393726,0015a82c2db000af6aaaf3ae2ecb0532,0,0.0
shipping_limit_date,str,93318,2020-04-09 22:35:08,2016-09-19 00:15:34,0,0.0
price,float64,5968,6735.0,0.85,0,0.0
freight_value,float64,6999,409.68,0.0,0,0.0


... (8 outputs left out)

===== sellers =====


,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
seller_id,str,3095,ffff564a4f9085cd26170f4732393726,0015a82c2db000af6aaaf3ae2ecb0532,0,0.0
seller_zip_code_prefix,int64,2246,99730,1001,0,0.0
seller_city,str,611,xaxim,04482255,0,0.0
seller_state,str,23,SP,AC,0,0.0


===== product_category_name_translation =====


,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
product_category_name,str,71,utilidades_domesticas,agro_industria_e_comercio,0,0.0
product_category_name_english,str,71,watches_gifts,agro_industry_and_commerce,0,0.0


## 1 - Table Orders

In [3]:
orders = df_dic["orders"]

print(orders.describe().round(2))
print("\n")
print(resume(orders))

                                order_id                       customer_id  \
count                              99441                             99441   
unique                             99441                             99441   
top     e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
freq                                   1                                 1   

       order_status order_purchase_timestamp    order_approved_at  \
count         99441                    99441                99281   
unique            8                    98875                90733   
top       delivered      2018-03-31 15:08:21  2018-02-27 04:31:10   
freq          96478                        3                    9   

       order_delivered_carrier_date order_delivered_customer_date  \
count                         97658                         96476   
unique                        81018                         95664   
top             2018-05-09 15:48:00           2018-05-14

### Conversion des colonnes en datetime

Les colonnes contenant des dates sont des `string` ce qui va empecher des opérations.

In [4]:
for col in [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]:
    orders[col] = pd.to_datetime(orders[col])

### Vérification de la cohérence des dates

J'ai trouvé des incohérences de dates dans le fichier d'origine

In [5]:
dates_tests = [
    ("order_approved_at", "order_purchase_timestamp"),
    ("order_delivered_carrier_date", "order_approved_at"),
    ("order_delivered_customer_date", "order_delivered_carrier_date"),
    ("order_estimated_delivery_date", "order_purchase_timestamp"),
]

for avant, apres in dates_tests:
    wrong_dates = (orders[avant] < orders[apres]).sum()
    print(f"Nombre de dates erronées pour {avant} < {apres} : {wrong_dates}")


Nombre de dates erronées pour order_approved_at < order_purchase_timestamp : 0
Nombre de dates erronées pour order_delivered_carrier_date < order_approved_at : 1359
Nombre de dates erronées pour order_delivered_customer_date < order_delivered_carrier_date : 23
Nombre de dates erronées pour order_estimated_delivery_date < order_purchase_timestamp : 0


### Ajout de colonnes imputee_*

Pour garder une trace des enregistrements modifiés on crée une colonne booléenne **imputee_none** pour les valeurs manquantes et **imputee_error** pour les valeurs incohérentes.

In [6]:
orders["imputee_none"] = (
    orders[
        [
            "order_delivered_customer_date",
            "order_delivered_carrier_date",
            "order_approved_at",
        ]
    ]
    .isna()
    .any(axis=1)
)

orders["imputee_error"] = (
    (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"])
    | (orders["order_delivered_carrier_date"] < orders["order_approved_at"])
    | (orders["order_approved_at"] < orders["order_purchase_timestamp"])
)

### Recherche des ecart moyens entres les colonnes

On va calculer les écarts seulements sur lignes avec le statut *delivered*

In [7]:
ecart_approved = (
    (orders["order_approved_at"] - orders["order_purchase_timestamp"])
    .where(orders["order_status"] == "delivered")
    .mean()
)

ecart_delivered_carrier = (
    (orders["order_delivered_carrier_date"] - orders["order_approved_at"])
    .where(orders["order_status"] == "delivered")
    .where(orders["order_delivered_carrier_date"] > orders["order_approved_at"])
    .mean()
)

ecart_delivered_customer = (
    (orders["order_delivered_customer_date"] - orders["order_delivered_carrier_date"])
    .where(orders["order_status"] == "delivered")
    .where(
        orders["order_delivered_customer_date"] > orders["order_delivered_carrier_date"]
    )
    .mean()
)

print(f"Écart entre l'approbation et l'achat : {ecart_approved}")
print(
    f"Écart entre l'approbation et la remise au transporteur : {ecart_delivered_carrier}"
)
print(
    f"Écart entre la remise au transporteur et la livraison au client : {ecart_delivered_customer}"
)

Écart entre l'approbation et l'achat : 0 days 10:16:36.361243
Écart entre l'approbation et la remise au transporteur : 2 days 20:28:55.066805
Écart entre la remise au transporteur et la livraison au client : 9 days 08:01:12.521625


### Remplacement des données manquantes par la moyenne

In [8]:
orders["order_approved_at"] = orders["order_approved_at"].fillna(
    orders["order_purchase_timestamp"] + ecart_approved
)

orders["order_delivered_carrier_date"] = orders["order_delivered_carrier_date"].fillna(
    orders["order_approved_at"] + ecart_delivered_carrier
)

orders["order_delivered_customer_date"] = orders[
    "order_delivered_customer_date"
].fillna(orders["order_delivered_carrier_date"] + ecart_delivered_customer)

### Correction incohérences des dates

In [9]:
orders["order_delivered_carrier_date"] = orders["order_delivered_carrier_date"].mask(
    orders["order_delivered_carrier_date"] < orders["order_approved_at"],
    (orders["order_approved_at"] + ecart_delivered_carrier),
)

orders["order_delivered_customer_date"] = orders["order_delivered_customer_date"].mask(
    orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"],
    (orders["order_delivered_carrier_date"] + ecart_delivered_customer),
)

In [10]:
# cellule de visualisation des données
orders.describe().round(2)
resume(orders)

,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
order_id,str,99441,fffe41c64501cc87c801fd61db3f6244,00010242fe8c5a6d1ba2dd792cb16214,0,0.0
customer_id,str,99441,ffffe8b65bbe3087b653a978c870db99,00012a2ce6f8dcda20d059ce98491703,0,0.0
order_status,str,8,unavailable,approved,0,0.0
order_purchase_timestamp,datetime64[us],98875,2018-10-17 17:30:18,2016-09-04 21:15:19,0,0.0
order_approved_at,datetime64[us],90893,2018-10-18 03:46:54.361243,2016-09-15 12:16:38,0,0.0
order_delivered_carrier_date,datetime64[us],83101,2018-10-21 00:15:49.428048,2016-10-07 06:55:35.066805,0,0.0
order_delivered_customer_date,datetime64[us],98605,2018-10-30 08:17:01.949673,2016-10-11 13:46:32,0,0.0
order_estimated_delivery_date,datetime64[us],459,2018-11-12 00:00:00,2016-09-30 00:00:00,0,0.0
imputee_none,bool,2,True,False,0,0.0
imputee_error,bool,2,True,False,0,0.0


### Vérifier la cohérence des dates

Il faut vérifier que les dates créers soient cohérentes, pas de livraisons avant la validation, etc.

In [11]:
dates_tests = [
    ("order_approved_at", "order_purchase_timestamp"),
    ("order_delivered_carrier_date", "order_approved_at"),
    ("order_delivered_customer_date", "order_delivered_carrier_date"),
    ("order_estimated_delivery_date", "order_purchase_timestamp"),
]

for avant, apres in dates_tests:
    wrong_dates = (orders[avant] < orders[apres]).sum()
    print(f"Nombre de dates erronées pour {avant} < {apres} : {wrong_dates}")


Nombre de dates erronées pour order_approved_at < order_purchase_timestamp : 0
Nombre de dates erronées pour order_delivered_carrier_date < order_approved_at : 0
Nombre de dates erronées pour order_delivered_customer_date < order_delivered_carrier_date : 0
Nombre de dates erronées pour order_estimated_delivery_date < order_purchase_timestamp : 0
